In [11]:
import os

for dirname, subdirs, filenames in os.walk('/kaggle/input'):
    if filenames:
        print(f"{dirname}  ({len(filenames)} files, e.g. {filenames[0]})")
    else:
        print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/rainbowpotatoes
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align/visible
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align/visible/test  (1013 files, e.g. FLIR_09725.jpeg)
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align/visible/train  (4129 files, e.g. FLIR_03409.jpeg)
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align/coco_annotations  (8 files, e.g. train_new_intid.json)
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align/thermal
/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align/thermal/test  (1013 files

In [12]:
!git clone --depth 1 https://github.com/RangiLyu/nanodet.git /kaggle/working/nanodet

# same fix needed locally -- torch._six was removed in modern torch
with open('/kaggle/working/nanodet/nanodet/data/collate.py', 'r') as f:
    content = f.read()
content = content.replace(
    'from torch._six import string_classes',
    'string_classes = (str, bytes)'
)
with open('/kaggle/working/nanodet/nanodet/data/collate.py', 'w') as f:
    f.write(content)

# strip the outdated torch/torchvision pins before installing -- Kaggle
# already has a working torch+CUDA setup preinstalled, same reasoning as locally
with open('/kaggle/working/nanodet/requirements.txt', 'r') as f:
    reqs = [line for line in f if not line.strip().lower().startswith(('torch', 'torchvision'))]
with open('/kaggle/working/nanodet/requirements.txt', 'w') as f:
    f.writelines(reqs)

!pip install -q -r /kaggle/working/nanodet/requirements.txt

import sys
sys.path.insert(0, '/kaggle/working/nanodet')

import nanodet
print("nanodet imported OK")

fatal: destination path '/kaggle/working/nanodet' already exists and is not an empty directory.
nanodet imported OK


In [13]:
import torch
import torch.nn as nn
from nanodet.util import cfg, load_config
from nanodet.model.arch import build_model

In [14]:
import torch
import torch.nn as nn
from nanodet.util import cfg, load_config
from nanodet.model.arch import build_model

# ---- build the architecture skeleton (must match Config A's checkpoint: num_classes=3) ----
MODEL_CONFIG_YAML = """
model:
  arch:
    name: NanoDetPlus
    detach_epoch: 10
    backbone:
      name: ShuffleNetV2
      model_size: 1.0x
      out_stages: [2,3,4]
      activation: LeakyReLU
      pretrain: False
    fpn:
      name: GhostPAN
      in_channels: [116, 232, 464]
      out_channels: 96
      kernel_size: 5
      num_extra_level: 1
      use_depthwise: True
      activation: LeakyReLU
    head:
      name: NanoDetPlusHead
      num_classes: 3
      input_channel: 96
      feat_channels: 96
      stacked_convs: 2
      kernel_size: 5
      strides: [8, 16, 32, 64]
      activation: LeakyReLU
      reg_max: 7
      norm_cfg:
        type: BN
      loss:
        loss_qfl:
          name: QualityFocalLoss
          use_sigmoid: True
          beta: 2.0
          loss_weight: 1.0
        loss_dfl:
          name: DistributionFocalLoss
          loss_weight: 0.25
        loss_bbox:
          name: GIoULoss
          loss_weight: 2.0
    aux_head:
      name: SimpleConvHead
      num_classes: 3
      input_channel: 192
      feat_channels: 192
      stacked_convs: 4
      strides: [8, 16, 32, 64]
      activation: LeakyReLU
      reg_max: 7
"""
with open('/kaggle/working/model_only_config.yml', 'w') as f:
    f.write(MODEL_CONFIG_YAML)

load_config(cfg, '/kaggle/working/model_only_config.yml')
base_model = build_model(cfg.model)

# ---- load Config A's trained weights -- specifically avg_model.*, since that's
# what NanoDet's own validation actually uses (confirmed from their source) ----
CONFIG_A_CHECKPOINT = "/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/checkpoint/model_best.ckpt"
ckpt = torch.load(CONFIG_A_CHECKPOINT, map_location="cpu")
state_dict = ckpt["state_dict"]

avg_model_weights = {
    k.replace("avg_model.", "", 1): v
    for k, v in state_dict.items()
    if k.startswith("avg_model.")
}
base_model.load_state_dict(avg_model_weights, strict=True)
print("Loaded avg_model weights from Config A checkpoint (strict=True, no mismatches)")

base_model.eval()
for p in base_model.parameters():
    p.requires_grad = False

# ---- thermal branch + fusion module (same design as YOLOv8n's) ----
class ThermalBranch(nn.Module):
    def __init__(self, out_channels, size="small"):
        super().__init__()
        mid = 16 if size == "small" else 32
        self.net = nn.Sequential(
            nn.Conv2d(1, mid, 3, 2, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid*2, 3, 2, 1), nn.BatchNorm2d(mid*2), nn.ReLU(inplace=True),
            nn.Conv2d(mid*2, out_channels, 3, 2, 1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, thermal):
        return self.net(thermal)


class ChannelAttentionFusion(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gate = nn.Sequential(
            nn.Linear(channels*2, channels//reduction), nn.ReLU(inplace=True),
            nn.Linear(channels//reduction, channels*2), nn.Sigmoid(),
        )
    def forward(self, rgb_feat, thermal_feat):
        b, c, h, w = rgb_feat.shape
        pooled = torch.cat([self.pool(rgb_feat).view(b,c), self.pool(thermal_feat).view(b,c)], dim=1)
        weights = self.gate(pooled)
        w_rgb, w_thermal = weights[:, :c].view(b,c,1,1), weights[:, c:].view(b,c,1,1)
        return rgb_feat * w_rgb + thermal_feat * w_thermal


# single-stage fusion: inject at backbone stage 0 only (116ch, 52x52 -- P3-equivalent)
thermal_branch = ThermalBranch(out_channels=116, size="small")
fusion_module = ChannelAttentionFusion(channels=116)

_current_thermal = {}

def fuse_hook(module, inputs, output):
    feat0, feat1, feat2 = output
    t_feat = thermal_branch(_current_thermal["x"])
    feat0 = fusion_module(feat0, t_feat)
    return (feat0, feat1, feat2)

base_model.backbone.register_forward_hook(fuse_hook)

trainable = sum(p.numel() for p in thermal_branch.parameters()) + sum(p.numel() for p in fusion_module.parameters())
frozen = sum(p.numel() for p in base_model.parameters())
print(f"trainable params: {trainable:,}  frozen params: {frozen:,}")

model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.
Loaded avg_model weights from Config A checkpoint (strict=True, no mismatches)
trainable params: 45,394  frozen params: 4,167,895


In [22]:
import cv2
import json
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

DATASET_ROOT = Path("/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align")
IMG_SIZE = 416

NORM_MEAN = np.array([103.53, 116.28, 123.675], dtype=np.float32).reshape(1, 1, 3)
NORM_STD = np.array([57.375, 57.12, 58.395], dtype=np.float32).reshape(1, 1, 3)


class PairedFlirDatasetNanoDet(Dataset):
    def __init__(self, split: str):
        assert split in ("train", "val")
        img_subdir = "train" if split == "train" else "test"
        ann_name = "train_new_intid.json" if split == "train" else "test_new_intid.json"

        self.rgb_dir = DATASET_ROOT / "visible" / img_subdir
        self.thermal_dir = DATASET_ROOT / "thermal" / img_subdir

        with open(DATASET_ROOT / "coco_annotations" / ann_name) as f:
            data = json.load(f)

        self.images_by_id = {img["id"]: img for img in data["images"]}
        self.anns_by_image = {}
        for ann in data["annotations"]:
            self.anns_by_image.setdefault(ann["image_id"], []).append(ann)

        self.image_ids = sorted(self.images_by_id.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images_by_id[img_id]
        file_name = img_info["file_name"]
        orig_w, orig_h = img_info["width"], img_info["height"]

        rgb = cv2.imread(str(self.rgb_dir / file_name))
        rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
        rgb = rgb.astype(np.float32)
        rgb = (rgb - NORM_MEAN) / NORM_STD
        rgb = rgb.transpose(2, 0, 1)

        thermal = cv2.imread(str(self.thermal_dir / file_name), cv2.IMREAD_GRAYSCALE)
        thermal = cv2.resize(thermal, (IMG_SIZE, IMG_SIZE))
        thermal = thermal.astype(np.float32) / 255.0
        thermal = thermal[np.newaxis, :, :]

        scale_x = IMG_SIZE / orig_w
        scale_y = IMG_SIZE / orig_h
        anns = self.anns_by_image.get(img_id, [])
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            x1, y1, x2, y2 = x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y
            boxes.append([x1, y1, x2, y2])
            labels.append(ann["category_id"])

        boxes = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
        labels = np.array(labels, dtype=np.int64) if labels else np.zeros((0,), dtype=np.int64)

        return {
            "rgb": rgb, "thermal": thermal, "boxes": boxes, "labels": labels,
            "img_id": img_id, "orig_w": orig_w, "orig_h": orig_h,
        }


def collate_fn(batch):
    rgb_batch = torch.stack([torch.from_numpy(item["rgb"]) for item in batch])
    thermal_batch = torch.stack([torch.from_numpy(item["thermal"]) for item in batch])
    gt_bboxes = [item["boxes"] for item in batch]
    gt_labels = [item["labels"] for item in batch]
    ids = [item["img_id"] for item in batch]
    heights = torch.tensor([IMG_SIZE] * len(batch))
    widths = torch.tensor([IMG_SIZE] * len(batch))

    gt_meta = {
        "img": rgb_batch, "gt_bboxes": gt_bboxes, "gt_labels": gt_labels,
        "gt_bboxes_ignore": None,
        "img_info": {"height": heights, "width": widths, "id": torch.tensor(ids)},
        "warp_matrix": [np.eye(3) for _ in range(len(batch))],
    }
    return gt_meta, thermal_batch


def eval_collate_fn(batch):
    rgb_batch = torch.stack([torch.from_numpy(item["rgb"]) for item in batch])
    thermal_batch = torch.stack([torch.from_numpy(item["thermal"]) for item in batch])
    gt_bboxes = [item["boxes"] for item in batch]
    gt_labels = [item["labels"] for item in batch]
    ids = [item["img_id"] for item in batch]

    orig_heights = torch.tensor([item["orig_h"] for item in batch])
    orig_widths = torch.tensor([item["orig_w"] for item in batch])

    warp_matrices = []
    for item in batch:
        scale_x = IMG_SIZE / item["orig_w"]
        scale_y = IMG_SIZE / item["orig_h"]
        warp_matrices.append(np.array([[scale_x, 0, 0], [0, scale_y, 0], [0, 0, 1]], dtype=np.float64))

    gt_meta = {
        "img": rgb_batch, "gt_bboxes": gt_bboxes, "gt_labels": gt_labels,
        "gt_bboxes_ignore": None,
        "img_info": {"height": orig_heights, "width": orig_widths, "id": torch.tensor(ids)},
        "warp_matrix": warp_matrices,
    }
    return gt_meta, thermal_batch


train_dataset = PairedFlirDatasetNanoDet("train")
val_dataset = PairedFlirDatasetNanoDet("val")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2)
eval_val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=eval_collate_fn, num_workers=2)

print(f"train: {len(train_dataset)} images, val: {len(val_dataset)} images")

gt_meta, thermal_batch = next(iter(train_loader))
print(f"img batch shape: {gt_meta['img'].shape}")
print(f"thermal batch shape: {thermal_batch.shape}")
print(f"num boxes in first image: {len(gt_meta['gt_bboxes'][0])}")

train: 4129 images, val: 1013 images
img batch shape: torch.Size([32, 3, 416, 416])
thermal batch shape: torch.Size([32, 1, 416, 416])
num boxes in first image: 12


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_model.to(device)
thermal_branch.to(device)
fusion_module.to(device)

Using device: cuda


ChannelAttentionFusion(
  (pool): AdaptiveAvgPool2d(output_size=1)
  (gate): Sequential(
    (0): Linear(in_features=232, out_features=14, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=14, out_features=232, bias=True)
    (3): Sigmoid()
  )
)

In [17]:
from pathlib import Path

EPOCHS = 30  # matching the epoch budget used for YOLOv8n's fusion configs -- a
             # small newly-added module converges much faster than Config A's
             # full 60-epoch fine-tune needed
CHECKPOINT_PATH = Path("/kaggle/working/config_b_nanodet_single_fusion_best.pt")

optimizer = torch.optim.AdamW(
    list(thermal_branch.parameters()) + list(fusion_module.parameters()),
    lr=1e-3,
)

best_val_loss = float("inf")
n_train_batches = len(train_loader)

for epoch in range(EPOCHS):
    base_model.set_epoch(epoch)  # NanoDet's own internal epoch tracking --
                                  # needed since forward_train checks self.epoch
                                  # against detach_epoch (10) to decide whether
                                  # the auxiliary head's gradients get detached
                                  # from the backbone-derived features

    thermal_branch.train()
    fusion_module.train()

    train_loss_total = 0.0
    for batch_num, (gt_meta, thermal_batch) in enumerate(train_loader):
        gt_meta["img"] = gt_meta["img"].to(device)
        thermal_batch = thermal_batch.to(device)
        _current_thermal["x"] = thermal_batch

        optimizer.zero_grad()
        head_out, loss, loss_states = base_model.forward_train(gt_meta)
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item()

        if batch_num % 20 == 0:
            print(f"  epoch {epoch+1}, batch {batch_num}/{n_train_batches}, running loss: {train_loss_total/(batch_num+1):.4f}")

    avg_train_loss = train_loss_total / len(train_loader)

    thermal_branch.eval()
    fusion_module.eval()

    val_loss_total = 0.0
    with torch.no_grad():
        for gt_meta, thermal_batch in val_loader:
            gt_meta["img"] = gt_meta["img"].to(device)
            thermal_batch = thermal_batch.to(device)
            _current_thermal["x"] = thermal_batch

            head_out, loss, loss_states = base_model.forward_train(gt_meta)
            val_loss_total += loss.item()

    avg_val_loss = val_loss_total / len(val_loader)
    print(f"epoch {epoch+1}/{EPOCHS}  train_loss={avg_train_loss:.4f}  val_loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "thermal_branch": thermal_branch.state_dict(),
            "fusion_module": fusion_module.state_dict(),
        }, CHECKPOINT_PATH)
        print(f"  saved new best checkpoint (val_loss={avg_val_loss:.4f})")

print(f"Training complete. Best val loss: {best_val_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


  epoch 1, batch 0/130, running loss: 3.5042
  epoch 1, batch 20/130, running loss: 2.0276
  epoch 1, batch 40/130, running loss: 1.8132
  epoch 1, batch 60/130, running loss: 1.7106
  epoch 1, batch 80/130, running loss: 1.6513
  epoch 1, batch 100/130, running loss: 1.6095
  epoch 1, batch 120/130, running loss: 1.5767
epoch 1/30  train_loss=1.5602  val_loss=1.7307
  saved new best checkpoint (val_loss=1.7307)
  epoch 2, batch 0/130, running loss: 1.5016
  epoch 2, batch 20/130, running loss: 1.4011
  epoch 2, batch 40/130, running loss: 1.3940
  epoch 2, batch 60/130, running loss: 1.3873
  epoch 2, batch 80/130, running loss: 1.3900
  epoch 2, batch 100/130, running loss: 1.3819
  epoch 2, batch 120/130, running loss: 1.3793
epoch 2/30  train_loss=1.3807  val_loss=1.7066
  saved new best checkpoint (val_loss=1.7066)
  epoch 3, batch 0/130, running loss: 1.4409
  epoch 3, batch 20/130, running loss: 1.3603
  epoch 3, batch 40/130, running loss: 1.3670
  epoch 3, batch 60/130, runnin

In [18]:
import sys, torch, torch.nn as nn
import cv2, json
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO

# ---- re-setup: needed since a fresh kernel loses installed packages too ----
sys.path.insert(0, '/kaggle/working/nanodet')
!pip install -q -r /kaggle/working/nanodet/requirements.txt

from nanodet.util import cfg, load_config
from nanodet.model.arch import build_model
from nanodet.evaluator import CocoDetectionEvaluator

# ---- rebuild model architecture and load Config A's frozen weights ----
load_config(cfg, '/kaggle/working/model_only_config.yml')
base_model = build_model(cfg.model)

CONFIG_A_CHECKPOINT = "/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/checkpoint/model_best.ckpt"
ckpt = torch.load(CONFIG_A_CHECKPOINT, map_location="cpu")
avg_model_weights = {
    k.replace("avg_model.", "", 1): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("avg_model.")
}
base_model.load_state_dict(avg_model_weights, strict=True)
base_model.eval()
for p in base_model.parameters():
    p.requires_grad = False

# ---- rebuild thermal branch + fusion module architecture ----
class ThermalBranch(nn.Module):
    def __init__(self, out_channels, size="small"):
        super().__init__()
        mid = 16 if size == "small" else 32
        self.net = nn.Sequential(
            nn.Conv2d(1, mid, 3, 2, 1), nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
            nn.Conv2d(mid, mid*2, 3, 2, 1), nn.BatchNorm2d(mid*2), nn.ReLU(inplace=True),
            nn.Conv2d(mid*2, out_channels, 3, 2, 1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )
    def forward(self, thermal):
        return self.net(thermal)


class ChannelAttentionFusion(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gate = nn.Sequential(
            nn.Linear(channels*2, channels//reduction), nn.ReLU(inplace=True),
            nn.Linear(channels//reduction, channels*2), nn.Sigmoid(),
        )
    def forward(self, rgb_feat, thermal_feat):
        b, c, h, w = rgb_feat.shape
        pooled = torch.cat([self.pool(rgb_feat).view(b,c), self.pool(thermal_feat).view(b,c)], dim=1)
        weights = self.gate(pooled)
        w_rgb, w_thermal = weights[:, :c].view(b,c,1,1), weights[:, c:].view(b,c,1,1)
        return rgb_feat * w_rgb + thermal_feat * w_thermal


thermal_branch = ThermalBranch(out_channels=116, size="small")
fusion_module = ChannelAttentionFusion(channels=116)

# ---- load the TRAINED fusion weights from your saved checkpoint (not random init) ----
FUSION_CHECKPOINT = "/kaggle/working/config_b_nanodet_single_fusion_best.pt"
fusion_ckpt = torch.load(FUSION_CHECKPOINT, map_location="cpu")
thermal_branch.load_state_dict(fusion_ckpt["thermal_branch"])
fusion_module.load_state_dict(fusion_ckpt["fusion_module"])
print("Loaded trained Config B fusion weights")

_current_thermal = {}

def fuse_hook(module, inputs, output):
    feat0, feat1, feat2 = output
    t_feat = thermal_branch(_current_thermal["x"])
    feat0 = fusion_module(feat0, t_feat)
    return (feat0, feat1, feat2)

base_model.backbone.register_forward_hook(fuse_hook)

# ---- device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)
thermal_branch.to(device)
fusion_module.to(device)
print(f"Using device: {device}")

# ---- rebuild val dataset/loader ----
DATASET_ROOT = Path("/kaggle/input/datasets/rainbowpotatoes/flir-aligned-nanodet/kaggle_upload/flir_align")
IMG_SIZE = 416
NORM_MEAN = np.array([103.53, 116.28, 123.675], dtype=np.float32).reshape(1, 1, 3)
NORM_STD = np.array([57.375, 57.12, 58.395], dtype=np.float32).reshape(1, 1, 3)


class PairedFlirDatasetNanoDet(Dataset):
    def __init__(self, split: str):
        assert split in ("train", "val")
        img_subdir = "train" if split == "train" else "test"
        ann_name = "train_new_intid.json" if split == "train" else "test_new_intid.json"
        self.rgb_dir = DATASET_ROOT / "visible" / img_subdir
        self.thermal_dir = DATASET_ROOT / "thermal" / img_subdir
        with open(DATASET_ROOT / "coco_annotations" / ann_name) as f:
            data = json.load(f)
        self.images_by_id = {img["id"]: img for img in data["images"]}
        self.anns_by_image = {}
        for ann in data["annotations"]:
            self.anns_by_image.setdefault(ann["image_id"], []).append(ann)
        self.image_ids = sorted(self.images_by_id.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.images_by_id[img_id]
        file_name = img_info["file_name"]
        orig_w, orig_h = img_info["width"], img_info["height"]

        rgb = cv2.imread(str(self.rgb_dir / file_name))
        rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
        rgb = rgb.astype(np.float32)
        rgb = (rgb - NORM_MEAN) / NORM_STD
        rgb = rgb.transpose(2, 0, 1)

        thermal = cv2.imread(str(self.thermal_dir / file_name), cv2.IMREAD_GRAYSCALE)
        thermal = cv2.resize(thermal, (IMG_SIZE, IMG_SIZE))
        thermal = thermal.astype(np.float32) / 255.0
        thermal = thermal[np.newaxis, :, :]

        scale_x = IMG_SIZE / orig_w
        scale_y = IMG_SIZE / orig_h
        anns = self.anns_by_image.get(img_id, [])
        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            x1, y1, x2, y2 = x * scale_x, y * scale_y, (x + w) * scale_x, (y + h) * scale_y
            boxes.append([x1, y1, x2, y2])
            labels.append(ann["category_id"])
        boxes = np.array(boxes, dtype=np.float32) if boxes else np.zeros((0, 4), dtype=np.float32)
        labels = np.array(labels, dtype=np.int64) if labels else np.zeros((0,), dtype=np.int64)

        return {"rgb": rgb, "thermal": thermal, "boxes": boxes, "labels": labels, "img_id": img_id}


def collate_fn(batch):
    rgb_batch = torch.stack([torch.from_numpy(item["rgb"]) for item in batch])
    thermal_batch = torch.stack([torch.from_numpy(item["thermal"]) for item in batch])
    gt_bboxes = [item["boxes"] for item in batch]
    gt_labels = [item["labels"] for item in batch]
    ids = [item["img_id"] for item in batch]
    heights = torch.tensor([IMG_SIZE] * len(batch))
    widths = torch.tensor([IMG_SIZE] * len(batch))
    gt_meta = {
        "img": rgb_batch, "gt_bboxes": gt_bboxes, "gt_labels": gt_labels,
        "gt_bboxes_ignore": None,
        "img_info": {"height": heights, "width": widths, "id": torch.tensor(ids)},
        "warp_matrix": [np.eye(3) for _ in range(len(batch))],
    }
    return gt_meta, thermal_batch


val_dataset = PairedFlirDatasetNanoDet("val")
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=2)
print(f"val: {len(val_dataset)} images")

# ---- evaluation ----
val_dataset.coco_api = COCO(DATASET_ROOT / "coco_annotations" / "test_new_intid.json")
val_dataset.class_names = ["car", "person", "bicycle"]
val_dataset.cat_ids = sorted(val_dataset.coco_api.getCatIds())

evaluator = CocoDetectionEvaluator(val_dataset)

all_results = {}
with torch.no_grad():
    for gt_meta, thermal_batch in val_loader:
        gt_meta["img"] = gt_meta["img"].to(device)
        thermal_batch = thermal_batch.to(device)
        _current_thermal["x"] = thermal_batch
        preds = base_model(gt_meta["img"])
        det_results = base_model.head.post_process(preds, gt_meta)
        all_results.update(det_results)

print("=== Config B, full val set ===")
eval_results = evaluator.evaluate(all_results, save_dir="/kaggle/working/")
print(eval_results)

model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.
Loaded trained Config B fusion weights
Using device: cuda
val: 1013 images
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
=== Config B, full val set ===
Loading and preparing results...
DONE (t=0.88s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=9.48s).
Accumulating evaluation results...
DONE (t=0.91s).
{'mAP': np.float64(5.2277331022826474e-05), 'AP_50': np.float64(0.0001834409658071781), 'AP_75': np.float64(4.0306912958027605e-07), 'AP_small': np.float64(1.4244843594201067e-07), 'AP_m': np.float64(6.031225830769527e-06), 'AP_l': np.float64(0.0004417671802289908)}


In [23]:
from pycocotools.coco import COCO
from nanodet.evaluator import CocoDetectionEvaluator

val_dataset.coco_api = COCO(DATASET_ROOT / "coco_annotations" / "test_new_intid.json")
val_dataset.class_names = ["car", "person", "bicycle"]
val_dataset.cat_ids = sorted(val_dataset.coco_api.getCatIds())

evaluator = CocoDetectionEvaluator(val_dataset)

thermal_branch.eval()
fusion_module.eval()

all_results = {}
with torch.no_grad():
    for gt_meta, thermal_batch in eval_val_loader:
        gt_meta["img"] = gt_meta["img"].to(device)
        thermal_batch = thermal_batch.to(device)
        _current_thermal["x"] = thermal_batch

        preds = base_model(gt_meta["img"])
        det_results = base_model.head.post_process(preds, gt_meta)
        all_results.update(det_results)

print("=== Config B, full val set ===")
eval_results = evaluator.evaluate(all_results, save_dir="/kaggle/working/")
print(eval_results)

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
=== Config B, full val set ===
Loading and preparing results...
DONE (t=0.87s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.66s).
Accumulating evaluation results...
DONE (t=0.92s).
{'mAP': np.float64(0.27316316454038286), 'AP_50': np.float64(0.5770901704889368), 'AP_75': np.float64(0.21531975777221224), 'AP_small': np.float64(0.1360527647644149), 'AP_m': np.float64(0.34496637842984135), 'AP_l': np.float64(0.42062071612823865)}


In [21]:
# 1. Confirm exactly one fusion hook is registered on the backbone
# (if Cell 2 got run more than once in this session, hooks stack up silently)
print("Number of hooks on backbone:", len(base_model.backbone._forward_hooks))

# 2. Look at raw prediction confidence for one real validation batch
thermal_branch.eval()
fusion_module.eval()

gt_meta, thermal_batch = next(iter(val_loader))
gt_meta["img"] = gt_meta["img"].to(device)
thermal_batch = thermal_batch.to(device)
_current_thermal["x"] = thermal_batch

with torch.no_grad():
    preds = base_model(gt_meta["img"])
    cls_scores, bbox_preds = preds.split([3, preds.shape[-1] - 3], dim=-1)
    scores = cls_scores.sigmoid()

print("max confidence score across whole batch:", scores.max().item())
print("mean confidence score:", scores.mean().item())

Number of hooks on backbone: 1
max confidence score across whole batch: 0.9609850645065308
mean confidence score: 0.018164411187171936
